In [ ]:
import sys
import xarray as xr
import numpy as np
import pandas as pd
import math
import glob
import yaml
import cartopy
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colorbar import Colorbar # different way to handle colorbar
import matplotlib.ticker as mticker
import cmocean.cm as cmo
import seaborn as sns

# cartopy
import cartopy.crs as ccrs
from cartopy.mpl.geoaxes import GeoAxes
import cartopy.feature as cfeature
import dask

# import personal modules
# Path to modules
sys.path.append('../modules')
# Import my modules
import global_vars
from utils import roundPartial, select_months_ds
from plotter import draw_basemap, plot_terrain, plot_arscale_cbar
from colorline import colorline
from trajectory_post_funcs import calculate_heatmaps_from_trajectories
import customcmaps as ccmaps

dask.config.set(**{'array.slicing.split_large_chunks': True})

In [ ]:
path_to_data = global_vars.path_to_data
path_to_out  = '../out/'       # output files (numerical results, intermediate datafiles) -- read & write
path_to_figs = '../figs/'      # figures

In [ ]:
start_mon = 11
end_mon = 4
## load PRISM watershed precip dataset to get list of HUC8s
fname = path_to_data + 'preprocessed/PRISM/PRISM_HUC8_CO_sp.nc'
PRISM = xr.open_dataset(fname)
## add water year to data as coordinate
water_year = (PRISM.date.dt.month >= 10) + PRISM.date.dt.year
PRISM.coords['water_year'] = water_year
HUC8_ID_lst = PRISM.HUC8.values ## get list of HUC8 IDs
PRISM = select_months_ds(PRISM, start_mon, end_mon, 'date')

# ## subset to ssn
# PRISM = select_months_ds(PRISM, start_mon, end_mon, 'date')
## for each HUC8, what is the total WY precipitation?
PRISM_WY = PRISM.prec.groupby(PRISM.water_year).sum(dim="date").sum('water_year')
PRISM_WY


## for each HUC8, what is the total WY top-decile precipitation?
PRISM_90 = PRISM.where(PRISM.extreme == 1, drop=True)
PRISM_90WY = PRISM_90.prec.groupby(PRISM_90.water_year).sum(dim="date").mean('water_year')
PRISM_90WY

In [ ]:
## use tARgetv4 AR dt for identifying AR at coast
varname = 'tARget'
thres = 0


In [ ]:
def calculate_WY_contribution(PRISM, HUC8_ID):
    PRISM = PRISM.sel(HUC8=HUC8_ID)
    
    fname = path_to_data + '/preprocessed/ERA5_trajectories/combined_extreme_AR/PRISM_HUC8_{0}.nc'.format(HUC8_ID)
    ds = xr.open_dataset(fname)
    ds['ar_scale'] = ds.ar_scale.fillna(0)
    ds = select_months_ds(ds, start_mon, end_mon, 'start_date')
    
    ## calculate the total top-decile precipitation in each WY
    extreme_days = PRISM.where(PRISM.extreme > 0, drop=True).date.values
    ext_prec = PRISM.sel(date = extreme_days)
    extreme_prec = ext_prec.prec.groupby(ext_prec.water_year).sum(dim="date")

    extreme_prec = extreme_prec.to_dataframe()
    extreme_prec = extreme_prec.rename(columns={"prec": "Total Precipitation"})
    
    ## calculate the AR-related top-decile prec
    extreme_AR = ds.sel(start_date = extreme_days)
    extreme_AR = extreme_AR.where(extreme_AR[varname] > thres, drop=True).start_date.values
    
    ## select those dates from the PRISM dataset
    tmp = PRISM.sel(date=extreme_AR)
    ## convert to pandas df
    tmp = tmp.prec.to_dataframe()
    
    extreme_ar_prec = tmp.groupby(["water_year"], dropna=False).sum("date")
    extreme_ar_prec = extreme_ar_prec.rename(columns={"prec": "AR Associated"})
    
    ## calculate the total contribution fraction
    # extreme_contr = (extreme_ar_prec / extreme_prec)*100
    
    ## put into a dataframe
    df = pd.concat([extreme_prec, extreme_ar_prec], axis=1)
    # df = pd.DataFrame({'Total Precipitation': extreme_prec.values,
    #                    'AR Associated': extreme_ar_prec.prec.values},
    #                   index=extreme_prec.water_year.values)

    df['Percent'] = (df['AR Associated']/df['Total Precipitation'])*100
    
    return df

In [ ]:
# HUC8_ID_lst = ['14050001', '13010001', '10190002', '11020001']
# HUC8_lbl_lst = ['Upper Yampa', 'Rio Grande Headwaters', 'Upper South Platte', 'Arkansas Headwaters']
HUC8_ID_lst = ['14050001', '14010004', '14020002', '14080101']
HUC8_lbl_lst = ['Upper Yampa', 'Roaring Fork', 'Upper Gunnison', 'Upper San Juan']
df_lst = []
for i, HUC8ID in enumerate(HUC8_ID_lst):
    df = calculate_WY_contribution(PRISM, HUC8ID)
    df['wy_str'] = df.index.astype(str).str[-2:]
    df_lst.append(df)

In [ ]:
df_lst[-1]

In [ ]:
# list of letters to append to titles
letter_lst = list(map(chr, range(97, 123)))
fig = plt.figure(figsize=(8,12))
fig.dpi = 300
fname = '../figs/time_series_extreme_NDJFMA'
fmt1 = 'png'

nrows = 4
ncols = 1
gs = fig.add_gridspec(nrows, ncols)
color_list = ['b', 'r', 'g']

for k, df in enumerate(df_lst):
    ## Initialize the matplotlib figure
    ax = fig.add_subplot(gs[k, 0])
    
    # Plot the total precip
    sns.set_color_codes("pastel")
    d1 = sns.barplot(x="wy_str", y="Total Precipitation", data=df, color='r', label="non-AR top-decile precipitation")
    
    # Plot the precipitation where ARs were involved
    sns.set_color_codes("muted")
    d2 = sns.barplot(x="wy_str", y="AR Associated", data=df, color='r', label="AR top-decile precipitation")
    
    # Add a legend and informative axis label
    ax.set(ylim=(0, 300), ylabel="top-decile prec (mm yr$^{-1}$)", xlabel="")
    sns.despine(left=True, bottom=True)
    #     ax.set_title(subtitles[i])

    # ## add min/max/mean/median lines
    # ax.axhline(y=df['AR Associated'].min(), color='k', linestyle=':')
    # ax.axhline(y=df['AR Associated'].max(), color='k', linestyle=':') 
    # ax.axhline(y=df['AR Associated'].mean(), color='k', linestyle='-')

    ## add average WY top-decile precipitation
    ax.axhline(y=PRISM_90WY.sel(HUC8=HUC8_ID_lst[k]).values, color='k', linestyle='-', zorder=1, alpha=0.5)

    yloc = df['Total Precipitation'].values
    xloc = np.arange(0, len(df), 1)
    proportion = df['Percent'].values
    for i, (x, prop) in enumerate(zip(xloc, proportion)):
        if prop > 0:
            plt.text(x=x-0.4,
                     y=yloc[i]+1,
                     s=f'{int(np.round(prop, 0))}%',
                     color="black",
                     fontsize=6,
                     fontweight="light", zorder=200)
    
    
    if k == 0:
        ax.legend(ncol=1, loc="upper right", frameon=True)
    else:
        ax.get_legend().remove()
    
    if k == 3:
        ax.set(xlabel="Year")
    else:
        # ax.tick_params(labelbottom=False)
        pass
    # d1.set_xticklabels(d1.get_xticklabels(), rotation=45)

    titlestring = '({0}) {1}'.format(letter_lst[k], HUC8_lbl_lst[k])
    ax.text(0.02, 1., titlestring, ha='left', va='top', transform=ax.transAxes, fontsize=11., backgroundcolor='white', zorder=101)

    
fig.savefig('%s.%s' %(fname, fmt1), bbox_inches='tight', dpi=fig.dpi)
plt.show()